# Prepare enviroment

In [1]:
import json
import random
import datetime

import numpy as np
import matplotlib.pyplot as plt

from pathlib import Path
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, Subset
from torchvision import transforms

from collections import defaultdict

from metrics import psnr, compute_all

In [2]:
TRAIN_DATA_DIRECTORY = Path("../original_data/DIV2K_train_HR")
VALIDATION_DATA_DIRECTORY = Path("../original_data/DIV2K_valid_HR")

TRAIN_PREPROCESSED_DATA_DIRECTORY = Path("preprocessed_data/DIV2K_train_HR")
TRAIN_CLEAN_DATA_DIRECTORY = TRAIN_PREPROCESSED_DATA_DIRECTORY / "clean"
TRAIN_NOISE_DATA_DIRECTORY = TRAIN_PREPROCESSED_DATA_DIRECTORY / "noisy"
TRAIN_DENOISED_DATA_DIRECTORY = TRAIN_PREPROCESSED_DATA_DIRECTORY / "denoised"

VALIDATION_PREPROCESSED_DATA_DIRECTORY = Path("preprocessed_data/DIV2K_valid_HR")
VALIDATION_CLEAN_DATA_DIRECTORY = VALIDATION_PREPROCESSED_DATA_DIRECTORY / "clean"
VALIDATION_NOISE_DATA_DIRECTORY = VALIDATION_PREPROCESSED_DATA_DIRECTORY / "noisy"
VALIDATION_DENOISED_DATA_DIRECTORY = VALIDATION_PREPROCESSED_DATA_DIRECTORY / "denoised"

In [3]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

print(f"Using device: {device}")

Using device: cuda:0


# Load dataset

In [4]:
class DenoisingDataset(Dataset):
    EXTENSIONS = {".png", ".jpg", ".jpeg", ".bmp", ".tiff", ".tif", ".webp"}

    def __init__(
        self,
        clean_dir: Path,
        noisy_dir: Path,
        clean_suffix: str = "_clean",
        noisy_suffix: str = "_noisy",
        transform=None,
        image_size: tuple | None = (256, 256),
        extensions: set | None = None,
        max_patches_per_image: int | None = None,
    ):
        self.clean_dir = clean_dir
        self.noisy_dir = noisy_dir
        self.clean_suffix = clean_suffix
        self.noisy_suffix = noisy_suffix
        self.transform = transform
        self.extensions = extensions or self.EXTENSIONS
        self.max_patches_per_image = max_patches_per_image

        self.pairs = self._build_pairs()

        if not self.pairs:
            raise RuntimeError(
                f"No matching image pairs found.\n"
                f"  clean_dir : {self.clean_dir}\n"
                f"  noisy_dir : {self.noisy_dir}\n"
                f"  suffixes  : '{clean_suffix}' / '{noisy_suffix}'"
            )

        resize = [transforms.Resize(image_size)] if image_size else []
        self.base_transform = transforms.Compose(resize + [transforms.ToTensor()])

    def _extract_image_id(self, stem: str) -> str | None:
        patch_marker = "_patch"
        idx = stem.find(patch_marker)

        return stem[:idx] if idx != -1 else None

    def _build_pairs(self) -> list[tuple[Path, Path]]:
        pairs = []
        patch_counts: dict[str, int] = {}

        for clean_path in sorted(self.clean_dir.iterdir()):
            if clean_path.suffix.lower() not in self.extensions:
                continue

            stem = clean_path.stem

            if not stem.endswith(self.clean_suffix):
                continue

            if self.max_patches_per_image is not None:
                image_id = self._extract_image_id(stem)

                if image_id is not None:
                    count = patch_counts.get(image_id, 0)

                    if count >= self.max_patches_per_image:
                        continue

                    patch_counts[image_id] = count + 1

            base = stem[: -len(self.clean_suffix)]
            noisy_name = base + self.noisy_suffix + clean_path.suffix
            noisy_path = self.noisy_dir / noisy_name

            if noisy_path.exists():
                pairs.append((noisy_path, clean_path))
            else:
                print(f"[WARNING] No noisy counterpart for: {clean_path.name}")

        return pairs

    def __len__(self) -> int:
        return len(self.pairs)

    def __getitem__(self, idx: int) -> dict[str, torch.Tensor]:
        noisy_path, clean_path = self.pairs[idx]

        noisy = Image.open(noisy_path).convert("RGB")
        clean = Image.open(clean_path).convert("RGB")

        noisy = self.base_transform(noisy)
        clean = self.base_transform(clean)

        if self.transform is not None:
            seed = torch.randint(0, 2**32, (1,)).item()

            torch.manual_seed(seed)
            noisy = self.transform(noisy)

            torch.manual_seed(seed)
            clean = self.transform(clean)

        return {"noisy": noisy, "clean": clean}

In [5]:
full_training_dataset = DenoisingDataset(
    clean_dir=TRAIN_CLEAN_DATA_DIRECTORY,
    noisy_dir=TRAIN_NOISE_DATA_DIRECTORY,
    max_patches_per_image=10,
)

In [6]:
pair = full_training_dataset[0]

noisy_image = pair["noisy"]
print(f"Noisy image shape: {noisy_image.shape}")

clean_image = pair["clean"]
print(f"Clean image shape: {clean_image.shape}")

Noisy image shape: torch.Size([3, 256, 256])
Clean image shape: torch.Size([3, 256, 256])


In [7]:
def split_by_image_id(dataset, val_ratio=0.2, seed=42):
    groups = defaultdict(list)

    for idx, (_, clean_path) in enumerate(dataset.pairs):
        image_id = dataset._extract_image_id(clean_path.stem)
        groups[image_id].append(idx)

    image_ids = sorted(groups.keys())

    rng = random.Random(seed)
    rng.shuffle(image_ids)

    split = int(len(image_ids) * (1 - val_ratio))
    train_ids = image_ids[:split]
    val_ids   = image_ids[split:]

    train_indices = [i for uid in train_ids for i in groups[uid]]
    val_indices   = [i for uid in val_ids   for i in groups[uid]]

    return Subset(dataset, train_indices), Subset(dataset, val_indices)

In [8]:
def get_image_ids(subset):
    dataset = subset.dataset

    return {
        dataset._extract_image_id(dataset.pairs[i][1].stem)
        for i in subset.indices
    }

In [9]:
train_dataset, val_dataset = split_by_image_id(full_training_dataset, val_ratio=0.2)

In [10]:
assert get_image_ids(train_dataset).isdisjoint(get_image_ids(val_dataset)), "Leakage!"

print(f"Train images: {len(get_image_ids(train_dataset))}")
print(f"Val images: {len(get_image_ids(val_dataset))}")
print(f"Train patches: {len(train_dataset)}")
print(f"Val patches: {len(val_dataset)}")

Train images: 640
Val images: 160
Train patches: 6400
Val patches: 1600


In [11]:
BATCH_SIZE = 4

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=4,
    pin_memory=True,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=4,
    pin_memory=True,
)

In [12]:
print(f"Train batches : {len(train_loader)}")
print(f"Val batches : {len(val_loader)}")

Train batches : 1600
Val batches : 400


In [13]:
batch = next(iter(train_loader))

print(f"Noisy batch shape: {batch['noisy'].shape}")
print(f"Clean batch shape: {batch['clean'].shape}")

Noisy batch shape: torch.Size([4, 3, 256, 256])
Clean batch shape: torch.Size([4, 3, 256, 256])


# Model architecture

In [14]:
class EAM(nn.Module):
    def __init__(self):
        super().__init__()

        # ── Multi-scale dilated branch ─────────────────────────────────
        self.conv1 = nn.Conv2d(64, 64, 3, padding=1, dilation=1)   # pad = dilation*(k-1)/2
        self.conv2 = nn.Conv2d(64, 64, 3, padding=2, dilation=2)

        self.conv3 = nn.Conv2d(64, 64, 3, padding=3, dilation=3)
        self.conv4 = nn.Conv2d(64, 64, 3, padding=4, dilation=4)

        # After concat the two dilated branches (128 ch) → 64 ch
        self.conv5 = nn.Conv2d(128, 64, 3, padding=1)

        # ── Residual block 1 ──────────────────────────────────────────
        self.conv6 = nn.Conv2d(64, 64, 3, padding=1)
        self.conv7 = nn.Conv2d(64, 64, 3, padding=1)

        # ── Residual block 2 ──────────────────────────────────────────
        self.conv8 = nn.Conv2d(64, 64, 3, padding=1)
        self.conv9 = nn.Conv2d(64, 64, 3, padding=1)
        self.conv10 = nn.Conv2d(64, 64, 1)

        # ── Channel attention (GAP → conv → sigmoid) ──────────────────
        self.conv11 = nn.Conv2d(64, 64, 3, padding=1)
        self.conv12 = nn.Conv2d(64, 64, 3, padding=1)

        self.relu = nn.ReLU(inplace=True)

    def forward(self, x):
        # ── Dilated feature extraction ────────────────────────────────
        b1 = self.relu(self.conv2(self.relu(self.conv1(x))))   # dilation 1→2
        b2 = self.relu(self.conv4(self.relu(self.conv3(x))))   # dilation 3→4

        cat = torch.cat([b1, b2], dim=1)                     # (B,128,H,W)
        conv3 = self.relu(self.conv5(cat))
        add1 = x + conv3                                      # residual

        # ── Residual block 1 ──────────────────────────────────────────
        conv4 = self.conv7(self.relu(self.conv6(add1)))
        add2 = self.relu(add1 + conv4)

        # ── Residual block 2 ──────────────────────────────────────────
        conv5 = self.conv10(self.relu(self.conv9(self.relu(self.conv8(add2)))))
        add3 = self.relu(add2 + conv5)

        # ── Channel attention ─────────────────────────────────────────
        gap = add3.mean(dim=(2, 3), keepdim=True)             # (B,64,1,1)
        attn = torch.sigmoid(self.conv12(self.relu(self.conv11(gap))))

        mul = attn * add3
        out = x + mul

        return out

In [15]:
class RIDNet(nn.Module):
    def __init__(self, num_eam: int = 4):
        super().__init__()

        self.head = nn.Conv2d(3, 64, 3, padding=1)

        self.eam_blocks = nn.Sequential(
            *[EAM() for _ in range(num_eam)]
        )

        self.tail = nn.Conv2d(64, 3, 3, padding=1)

    def forward(self, x):
        feat = self.head(x)
        feat = self.eam_blocks(feat)
        out  = self.tail(feat)

        return x + out                  # global residual skip

In [16]:
model = RIDNet(num_eam=3).to(device)

In [17]:
total = sum(p.numel() for p in model.parameters())

print(f"Parameters: {total:,}")

Parameters: 1,345,219


# Training

In [18]:
LEARNING_RATE = 1e-3
LR_STEP = 5
LR_GAMMA = 0.5

EPOCHS_NUM = 10
PATIENCE = 3

In [19]:
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
criterion = torch.nn.MSELoss()
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=LR_STEP, gamma=LR_GAMMA)

In [20]:
def get_timestamp() -> str:
    ct = datetime.datetime.now()

    return ct.strftime("%Y-%m-%d_%H-%M-%S")

In [21]:
CURRENT_RUN_DIRECTORY = Path(f"runs/{get_timestamp()}")
CURRENT_RUN_DIRECTORY.mkdir(parents=True, exist_ok=True)

In [22]:
def save_checkpoint(state: dict, path: Path):
    path.parent.mkdir(parents=True, exist_ok=True)
    torch.save(state, path)

    print(f"Checkpoint saved to {path}")

In [ ]:
history: dict[str, list[float]] = {
    "train_loss": [],
    "val_loss": [],
    "train_psnr": [],
    "val_psnr": [],
    "lr": [],
}

best_val_loss = float("inf")
patience_counter = 0
best_model_state = None

for epoch in range(EPOCHS_NUM):
    model.train()

    train_losses = []
    train_psnrs = []

    for batch in train_loader:
        noisy = batch["noisy"].to(device)
        clean = batch["clean"].to(device)

        pred = model(noisy)
        loss = criterion(pred, clean)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        train_losses.append(loss.item())
        train_psnrs.append(psnr(pred, clean))

    train_loss = np.mean(train_losses)
    train_psnr = np.mean(train_psnrs)

    model.eval()

    val_losses = []
    val_psnrs = []

    with torch.no_grad():
        for batch in val_loader:
            noisy = batch["noisy"].to(device)
            clean = batch["clean"].to(device)

            pred = model(noisy)
            loss = criterion(pred, clean)

            val_losses.append(loss.item())
            val_psnrs.append(psnr(pred, clean))

    val_loss = np.mean(val_losses)
    val_psnr = np.mean(val_psnrs)

    scheduler.step()

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["train_psnr"].append(train_psnr)
    history["val_psnr"].append(val_psnr)
    history["lr"].append(optimizer.param_groups[0]["lr"])

    print(
        f"Epoch {epoch+1} | "
        f"Train Loss: {train_loss:.4f} PSNR: {train_psnr:.4f} | "
        f"Val Loss: {val_loss:.4f} PSNR: {val_psnr:.4f} | "
        f"LR: {optimizer.param_groups[0]['lr']:.6f}"
    )

    if val_loss < best_val_loss:
        best_val_loss = val_loss

        best_model_state = model.state_dict()
        save_checkpoint(
            {
                "epoch": epoch,
                "model": best_model_state,
                "optimizer": optimizer.state_dict(),
                "scheduler": scheduler.state_dict(),
                "best_val_loss": best_val_loss,
            },
            path=Path(f"{CURRENT_RUN_DIRECTORY}/best_model.pth"),
        )

        patience_counter = 0
    else:
        patience_counter += 1

        if patience_counter >= PATIENCE:
            print(f"Early stopping at epoch {epoch+1}")
            break

print("Training finished.")

Epoch 1 | Train Loss: 0.0009 PSNR: 35.4908 | Val Loss: 0.0002 PSNR: 37.2309 | LR: 0.001000
Checkpoint saved to runs/2026-03-21_23-03-36/best_model.pth
Epoch 2 | Train Loss: 0.0002 PSNR: 37.5683 | Val Loss: 0.0002 PSNR: 38.3639 | LR: 0.001000
Checkpoint saved to runs/2026-03-21_23-03-36/best_model.pth
Epoch 3 | Train Loss: 0.0002 PSNR: 38.0391 | Val Loss: 0.0002 PSNR: 38.7310 | LR: 0.001000
Checkpoint saved to runs/2026-03-21_23-03-36/best_model.pth
Epoch 4 | Train Loss: 0.0002 PSNR: 38.4715 | Val Loss: 0.0002 PSNR: 38.0443 | LR: 0.001000
Epoch 5 | Train Loss: 0.0001 PSNR: 38.7214 | Val Loss: 0.0001 PSNR: 39.0871 | LR: 0.000500
Checkpoint saved to runs/2026-03-21_23-03-36/best_model.pth
Epoch 6 | Train Loss: 0.0001 PSNR: 39.1158 | Val Loss: 0.0001 PSNR: 39.5342 | LR: 0.000500
Checkpoint saved to runs/2026-03-21_23-03-36/best_model.pth
Epoch 7 | Train Loss: 0.0001 PSNR: 39.5629 | Val Loss: 0.0001 PSNR: 39.9295 | LR: 0.000500
Checkpoint saved to runs/2026-03-21_23-03-36/best_model.pth


In [ ]:
history_path = Path(f"{CURRENT_RUN_DIRECTORY}/history.json")

with open(history_path, "w") as f:
    json.dump(history, f, indent=4, sort_keys=True)

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(history["train_loss"], label="Train Loss")
plt.plot(history["val_loss"], label="Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training vs Validation Loss")
plt.legend()
plt.grid(True)
plt.savefig(f"{CURRENT_RUN_DIRECTORY}/loss.png", dpi=300, bbox_inches="tight")

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(history["train_psnr"], label="Train PSNR")
plt.plot(history["val_psnr"], label="Validation PSNR")
plt.xlabel("Epoch")
plt.ylabel("PSNR")
plt.title("Training vs Validation PSNR")
plt.legend()
plt.grid(True)
plt.savefig(f"{CURRENT_RUN_DIRECTORY}/psnr.png", dpi=300, bbox_inches="tight")

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(history["lr"], label="Learning Rate")
plt.xlabel("Epoch")
plt.ylabel("Learning Rate")
plt.title("Learning Rate per Epoch")
plt.legend()
plt.grid(True)
plt.savefig(f"{CURRENT_RUN_DIRECTORY}/lr.png", dpi=300, bbox_inches="tight")

# Evaluate model

## Load testing dataset

In [ ]:
test_dataset = DenoisingDataset(
    clean_dir=VALIDATION_CLEAN_DATA_DIRECTORY,
    noisy_dir=VALIDATION_NOISE_DATA_DIRECTORY,
    max_patches_per_image=10,
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=4,
    pin_memory=True,
)

In [ ]:
print(f"Test patches: {len(test_dataset)}")
print(f"Test batches: {len(test_loader)}")

## Load model

In [ ]:
def load_model(path: Path, model: nn.Module):
    ckpt = torch.load(path, map_location="cpu", weights_only=False)
    model.load_state_dict(ckpt["model"])

    print(f"Loaded model from {path}")

    return model

In [ ]:
model = load_model(CURRENT_RUN_DIRECTORY / "best_model.pth", model).to(device)

# model_path = Path("runs/2026-03-21_18-02-48/best_model.pth")
# model = load_model(model_path, model).to(device)

## Calculate metrics

In [ ]:
model.eval()

baseline_losses  = []
baseline_metrics = []

model_losses  = []
model_metrics = []

with torch.no_grad():
    for batch_idx, batch in enumerate(test_loader):
        noisy = batch["noisy"].to(device)
        clean = batch["clean"].to(device)

        baseline_losses.append(criterion(noisy, clean).item())
        baseline_metrics.append(compute_all(noisy, clean))

        pred = model(noisy).clamp(0, 1)

        model_losses.append(criterion(pred, clean).item())
        model_metrics.append(compute_all(pred, clean))

In [ ]:
def mean_over_batches(losses, metrics):
    metric_keys = metrics[0].keys()

    return {
        "Loss": sum(losses) / len(losses),
        **{
            k: sum(m[k] for m in metrics) / len(metrics)
            for k in metric_keys
        },
    }

In [ ]:
baseline_results = mean_over_batches(baseline_losses, baseline_metrics)
model_results = mean_over_batches(model_losses, model_metrics)

In [ ]:
baseline_path = Path(f"{CURRENT_RUN_DIRECTORY}/baseline_metrics.json")

with open(baseline_path, "w") as f:
    json.dump(baseline_results, f, indent=4, sort_keys=True)

In [ ]:
model_results_path = Path(f"{CURRENT_RUN_DIRECTORY}/model_metrics.json")

with open(model_results_path, "w") as f:
    json.dump(model_results, f, indent=4, sort_keys=True)

In [ ]:
print("Original clean-noisy images pairs:")

for key, value in baseline_results.items():
    arrow = "↓ lower is better" if key in ("SNE", "LPIPS", "Loss") else "↑ higher is better"

    print(f"{key}: {value:.4f}       {arrow}")

In [ ]:
print("RIDNet results:")

for key, value in model_results.items():
    arrow = "↓ lower is better" if key in ("SNE", "LPIPS", "Loss") else "↑ higher is better"

    print(f"{key}: {value:.4f}       {arrow}")

# Evaluate denoise bilateral algorithm

## Load denoised data

In [ ]:
denoised_dataset = DenoisingDataset(
    clean_dir=VALIDATION_CLEAN_DATA_DIRECTORY,
    noisy_dir=VALIDATION_DENOISED_DATA_DIRECTORY,
    noisy_suffix="_denoised",
    max_patches_per_image=10,
)

denoised_loader = DataLoader(
    denoised_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=4,
    pin_memory=True,
)

In [ ]:
print(f"Denoised patches: {len(denoised_dataset)}")
print(f"Denoised batches : {len(denoised_loader)}")

## Calculate metrics

In [ ]:
model.eval()

denoised_losses  = []
denoised_metrics = []

with torch.no_grad():
    for batch_idx, batch in enumerate(denoised_loader):
        noisy = batch["noisy"].to(device)
        clean = batch["clean"].to(device)

        denoised_losses.append(criterion(noisy, clean).item())
        denoised_metrics.append(compute_all(noisy, clean))

In [ ]:
denoised_results = mean_over_batches(denoised_losses, denoised_metrics)

In [ ]:
denoised_path = Path(f"{CURRENT_RUN_DIRECTORY}/denoised_metrics.json")

with open(denoised_path, "w") as f:
    json.dump(denoised_results, f, indent=4, sort_keys=True)

In [ ]:
print("Denoise bilateral results:")

for key, value in denoised_results.items():
    arrow = "↓ lower is better" if key in ("SNE", "LPIPS", "Loss") else "↑ higher is better"

    print(f"{key}: {value:.4f}       {arrow}")